# Data Case Study Starter: Sector Volatility

Prepares you for **Data Case Study on Thursday, October 22**.

**Question.** Which sector in the synthetic universe is the most volatile, and by how much?

Steps 1 and 2 are provided. Steps 3 to 5 are yours: each has a cell marked `# TODO (member)`. The reference solution is in the **Solutions** section at the end. Attempt each step before you scroll down.

Before you start, run `python data/make_data.py` from the repository root once.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

WINDOW = 21                  # trading days in the rolling window, about one month
TRADING_DAYS_PER_YEAR = 252

# Find the repository root whether the notebook runs from pandas/ or from the root.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "data" / "make_data.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the repository.")
PRICES_CSV = ROOT / "data" / "prices.csv"
TICKERS_CSV = ROOT / "data" / "tickers.csv"
CHART_PATH = ROOT / "pandas" / "output" / "sector_volatility.png"
if not PRICES_CSV.exists():
    raise FileNotFoundError("data/prices.csv not found. Run python data/make_data.py first.")

## Step 1: Load prices with parsed dates (provided)

In [ ]:
prices = pd.read_csv(PRICES_CSV, parse_dates=["trade_date"])
prices = prices.sort_values(["ticker", "trade_date"]).reset_index(drop=True)
tickers = pd.read_csv(TICKERS_CSV)

print(prices.dtypes)
prices.head()

## Step 2: Daily log returns per ticker (provided)

`log_return = log(close_t / close_{t-1})`, computed within each ticker. Grouping matters: without it, the first return of each ticker would use the last close of the previous ticker. The first row of each ticker is NaN.

In [ ]:
returns = prices.copy()
returns["log_return"] = np.log(returns["close"]).groupby(returns["ticker"]).diff()
returns.groupby("ticker")["log_return"].describe()

## Step 3: 21-day rolling volatility, annualized

Add a column `rolling_vol` to a copy of `returns`: the standard deviation of `log_return` over the last 21 rows of the same ticker, multiplied by `sqrt(252)`. Store the result in `vol`.

In [ ]:
# TODO (member): compute rolling volatility per ticker.
# Hint: returns.groupby("ticker")["log_return"].transform(lambda s: s.rolling(WINDOW).std())
vol = None  # replace with your DataFrame

## Step 4: Average volatility by sector

Merge `vol` with `tickers` on `ticker`, drop rows where `rolling_vol` is NaN, and average `rolling_vol` by sector. Store a DataFrame with columns `sector` and `avg_vol`, sorted from most to least volatile, in `by_sector`.

In [ ]:
# TODO (member): merge, groupby("sector"), sort.
by_sector = None  # replace with your DataFrame

## Step 5: One labeled chart

Plot `avg_vol` by sector. Give it a title that states the finding, an axis label with units, and readable category names. Save it to `CHART_PATH` (create the folder first with `CHART_PATH.parent.mkdir(parents=True, exist_ok=True)`).

In [ ]:
# TODO (member): build and save one chart from by_sector.


---

# Solutions

Attempt steps 3 to 5 before reading this section. These cells overwrite `vol` and `by_sector` with the reference results.

### Solution: Step 3

In [ ]:
vol = returns.copy()
rolling_std = vol.groupby("ticker")["log_return"].transform(lambda s: s.rolling(WINDOW).std())
vol["rolling_vol"] = rolling_std * np.sqrt(TRADING_DAYS_PER_YEAR)
vol.groupby("ticker")["rolling_vol"].last().map("{:.1%}".format)

### Solution: Step 4

In [ ]:
merged = vol.merge(tickers, on="ticker", how="left", validate="many_to_one")
by_sector = (
    merged.dropna(subset=["rolling_vol"])
    .groupby("sector", as_index=False)["rolling_vol"]
    .mean()
    .rename(columns={"rolling_vol": "avg_vol"})
    .sort_values("avg_vol", ascending=False)
    .reset_index(drop=True)
)
by_sector.style.format({"avg_vol": "{:.1%}"})

### Solution: Step 5

In [ ]:
data = by_sector.sort_values("avg_vol")  # largest bar on top
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(data["sector"], data["avg_vol"] * 100, color="#2a78d6", height=0.6)
ax.bar_label(bars, labels=[f"{v:.1%}" for v in data["avg_vol"]], padding=4, color="#52514e")
top = by_sector.iloc[0]
ax.set_title(f"{top['sector']} is the most volatile sector ({top['avg_vol']:.1%} annualized)",
             loc="left", fontsize=11)
ax.set_xlabel("Average 21-day volatility, annualized (%)")
ax.set_xlim(0, data["avg_vol"].max() * 100 * 1.2)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", color="#e5e5e2", linewidth=0.8)
ax.set_axisbelow(True)
fig.text(0.01, 0.01, "Source: synthetic FQE data (data/make_data.py), 2024-01-02 to 2025-12-05.",
         fontsize=7, color="#52514e")
fig.tight_layout(rect=(0, 0.04, 1, 1))
CHART_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(CHART_PATH, dpi=150)
plt.show()

### The finding, with a number

In [ ]:
top, bottom = by_sector.iloc[0], by_sector.iloc[-1]
print(f"{top['sector']} has the highest average volatility at {top['avg_vol']:.1%}, "
      f"{top['avg_vol'] / bottom['avg_vol']:.1f} times {bottom['sector']} at {bottom['avg_vol']:.1%}.")